# Delaware PUMS Calibration — OTS 3.0 Local Verification

Validates the Chile–Barbados cumulant / ICR / sheaf pipeline against Delaware ACS PUMS person-level data (`psam_p10.csv`).  
Data source: U.S. Census Bureau ACS 1-Year PUMS, Delaware (State FIPS 10).  

**Sections**
1. Data Inspection & Variable Mapping
2. Chile-Barbados Interval — cumulant decomposition on equivalised household income
3. ICR Proxy Scaffolding — structural skeleton for future SCF/CEX integration
4. Sheaf Gluing — cocycle obstruction check across racial subgroups

In [1]:
import pathlib
import sys
import numpy as np
import pandas as pd
import torch
import scipy.stats as stats

# Locate project root by walking up until pyproject.toml is found
ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists(), f"Could not locate project root from {pathlib.Path.cwd()}"
sys.path.insert(0, str(ROOT))

from structural_impedance.cumulant import cross_cumulant_residual_perK, admit_per_component
from structural_impedance.sheaf_gluing import sheaf_status_and_kappa

DATA_PATH = ROOT / "data" / "pums" / "psam_p10.csv"
assert DATA_PATH.exists(), f"PUMS file not found: {DATA_PATH}"
print(f"ROOT       : {ROOT}")
print(f"DATA_PATH  : {DATA_PATH}")

ROOT       : /Users/alanevans/OTS_3.0_1160
DATA_PATH  : /Users/alanevans/OTS_3.0_1160/data/pums/psam_p10.csv


---
## Section 1 — Data Inspection & Variable Mapping

In [2]:
# Cell 1.1 — Load and inspect key columns
KEY_COLS = ["SERIALNO", "AGEP", "PINCP", "PERNP", "RAC1P", "SCHL", "RELSHIPP", "HISP"]

df = pd.read_csv(DATA_PATH)
print("=== dtypes ===")
print(df[KEY_COLS].dtypes)
print()
print("=== descriptive stats ===")
print(df[["PINCP", "PERNP", "RAC1P", "SCHL", "AGEP", "RELSHIPP"]].describe())
print()
print(f"Total persons : {len(df):,}")
print(f"Unique HH IDs : {df['SERIALNO'].nunique():,}")

=== dtypes ===
SERIALNO     object
AGEP          int64
PINCP       float64
PERNP       float64
RAC1P         int64
SCHL        float64
RELSHIPP      int64
HISP          int64
dtype: object

=== descriptive stats ===
               PINCP          PERNP        RAC1P         SCHL         AGEP  \
count    8680.000000    8560.000000  9876.000000  9670.000000  9876.000000   
mean    57385.176498   37838.103738     2.247671    17.147363    47.938437   
std     74145.906082   65785.673361     2.532267     5.170013    24.194659   
min     -3100.000000   -8600.000000     1.000000     1.000000     0.000000   
25%     14100.000000       0.000000     1.000000    16.000000    28.000000   
50%     40000.000000   10000.000000     1.000000    19.000000    53.000000   
75%     74000.000000   55000.000000     2.000000    21.000000    68.000000   
max    931900.000000  593000.000000     9.000000    24.000000    95.000000   

          RELSHIPP  
count  9876.000000  
mean     22.858141  
std       4.354265

In [3]:
# Cell 1.2 — Recode RAC1P and SCHL; verify counts

# RAC1P: 1=White alone, 2=Black/AA alone, 3-9=Other
race_map = {1: "White", 2: "Black"}
df["race_simple"] = df["RAC1P"].map(race_map).fillna("Other")

# SCHL: 1-24 ordinal ACS scale
def schl_to_level(s):
    if pd.isna(s) or s <= 1:
        return "No schooling"
    elif s <= 15:
        return "Some HS"
    elif s == 16:
        return "HS diploma"
    elif s <= 20:
        return "Some college"
    elif s == 21:
        return "Associate"
    elif s == 22:
        return "Bachelor"
    else:
        return "Grad+"

df["educ_level"] = df["SCHL"].apply(schl_to_level)

print("=== RAC1P \u2192 race_simple ===")
print(df["race_simple"].value_counts())
print()
print("=== SCHL \u2192 educ_level ===")
print(df["educ_level"].value_counts())

=== RAC1P → race_simple ===
White    6877
Other    1632
Black    1367
Name: race_simple, dtype: int64

=== SCHL → educ_level ===
Some college    2637
HS diploma      2056
Associate       1777
Some HS         1619
Bachelor         984
No schooling     431
Grad+            372
Name: educ_level, dtype: int64


**Mapping verification.**

| Variable | Source | Recode |
|---|---|---|
| Income | `PINCP` (total personal income, signed) | Raw; sign preserved |
| Race | `RAC1P` | 1→White, 2→Black, else→Other |
| Education | `SCHL` | 7-level ordinal grouping |
| Age | `AGEP` | Raw integer years |
| Household | `SERIALNO` | Group key for HH aggregation |
| Householder | `RELSHIPP == 20` | Reference person per ACS 2019+ encoding |

**Absent wealth variables:** `VALP` (property value) and `TEN` (tenure) are not present in the person-level PUMS file; they live in the household-level file (`psam_h10.csv`). This run uses equivalised household income as the welfare proxy. The full wealth-deficit model will join on `SERIALNO` once the household file is loaded.

---
## Section 2 — Chile-Barbados Interval

**Wealth proxy note:** PUMS person-level lacks `VALP`/`TEN`. Income (`PINCP`) is used as the cumulant analysis input. The purpose here is to validate the cumulant decomposition pipeline against real distributions — not yet the full wealth deficit.

The Chile-Barbados framework identifies distributional impedance: do Black and White household incomes show structural cross-cumulant divergence beyond mean/variance differences? The `cross_cumulant_residual_perK` function (axioms §0.5; Sturmfels & Zwiernik arXiv:1011.1722) zeros the within-group blocks of the joint cumulant tensor and returns the cross-block dependence signature.

In [4]:
# Cell 2.1 — Household aggregation and equivalised income

# Aggregate to household level: sum PINCP, count members
hh = df.groupby("SERIALNO").agg(
    hh_income=("PINCP", "sum"),
    hh_size=("PINCP", "count"),
    hh_race_fallback=("race_simple", "first"),
).reset_index()

# Override race with householder's race (RELSHIPP == 20 is the ACS reference person)
householders = (
    df[df["RELSHIPP"] == 20][["SERIALNO", "race_simple"]]
    .rename(columns={"race_simple": "hh_race_ref"})
)
hh = hh.merge(householders, on="SERIALNO", how="left")
# Group-quarters households (RELSHIPP=38 etc.) lack RELSHIPP=20; fall back to first member
hh["hh_race_ref"] = hh["hh_race_ref"].fillna(hh["hh_race_fallback"])

# Equivalised income: OECD modified scale \u2248 income / sqrt(household_size)
hh["equiv_income"] = hh["hh_income"] / np.sqrt(hh["hh_size"].clip(lower=1))

# Nucleation threshold: equivalised income > $30,000
hh_above = hh[hh["equiv_income"] > 30_000].copy()

print("=== Households above nucleation threshold ($30k equiv) ===")
print(hh_above["hh_race_ref"].value_counts())
print()
print(hh_above.groupby("hh_race_ref")["equiv_income"].describe())

=== Households above nucleation threshold ($30k equiv) ===
White    2693
Black     404
Other     388
Name: hh_race_ref, dtype: int64

              count           mean           std           min           25%  \
hh_race_ref                                                                    
Black         404.0   78999.956565  61287.857266  30476.302269  47040.535204   
Other         388.0  104851.275276  86417.234019  30022.213998  53510.305666   
White        2693.0   99557.126001  80225.724221  30022.213998  53457.272658   

                      50%            75%            max  
hh_race_ref                                              
Black        61304.084015   88804.919121  622961.074225  
Other        76438.243046  123849.752725  659023.520066  
White        76650.375081  113702.770415  926946.279457  


In [5]:
# Cell 2.2 — Cross-cumulant residual: Black vs White income distributions
#
# cross_cumulant_residual_perK concatenates X and Y along the feature axis (dim=1),
# forming a joint [n, d_x+d_y] sample. Both groups must have equal n.
# Equalize via min(n_B, n_W) random draw (seed 42 for reproducibility).

black_inc = hh_above[hh_above["hh_race_ref"] == "Black"]["equiv_income"].values
white_inc = hh_above[hh_above["hh_race_ref"] == "White"]["equiv_income"].values

if len(black_inc) == 0 or len(white_inc) == 0:
    raise RuntimeError(
        "One or both subgroups are empty above the nucleation threshold. "
        "Check PINCP and RELSHIPP column values."
    )

n = min(len(black_inc), len(white_inc))
rng = np.random.default_rng(42)
b_sample = rng.choice(black_inc, size=n, replace=False)
w_sample = rng.choice(white_inc, size=n, replace=False)

# [n, 1] tensors — 1D income feature (d=1)
X = torch.tensor(b_sample, dtype=torch.float64).unsqueeze(1)
Y = torch.tensor(w_sample, dtype=torch.float64).unsqueeze(1)

k3_residual = cross_cumulant_residual_perK(X, Y, k_order=3)  # [d_x + d_y] = [2]
k4_residual = cross_cumulant_residual_perK(X, Y, k_order=4)  # [2]
norm_k3 = float(k3_residual.norm())
norm_k4 = float(k4_residual.norm())

print(f"n_black (raw)    : {len(black_inc):,}")
print(f"n_white (raw)    : {len(white_inc):,}")
print(f"equalized n      : {n:,}")
print()
print(f"\u03ba_3 residual     : {k3_residual.tolist()}")
print(f"\u03ba_4 residual     : {k4_residual.tolist()}")
print()
print(f"||\u03ba_3||          : {norm_k3:.6f}")
print(f"||\u03ba_4||          : {norm_k4:.6f}")

n_black (raw)    : 404
n_white (raw)    : 2,693
equalized n      : 404

κ_3 residual     : [48865379835638.62, 67675545538618.38]
κ_4 residual     : [5.145197748302492e+19, 8.017727800539816e+19]

||κ_3||          : 83473377854444.390625
||κ_4||          : 95266478339805380608.000000


In [6]:
# Cell 2.3 — Per-component admission gate (axioms §0.5.1)
#
# ADMIT iff ||\u03ba_3|| > \u03c4_3 AND ||\u03ba_4|| > \u03c4_4 independently.
# A high \u03ba_3 may NOT mask a zero \u03ba_4 (anti-Goodhart per-component gate).

admit, diag = admit_per_component(X, Y, tau_3=1e-3, tau_4=1e-3)

print(f"admit                : {admit}")
print(f"blocking_component   : {diag.get('blocking_component')}")
print()
print("full diagnostic:")
for k, v in diag.items():
    print(f"  {k:<22}: {v}")

admit                : True
blocking_component   : None

full diagnostic:
  ||k3||                : 83473377854444.39
  ||k4||                : 9.526647833980538e+19
  tau_3                 : 0.001
  tau_4                 : 0.001
  blocking_component    : None


---
## Section 3 — Intercept-to-Compound Ratio (ICR) Proxy Scaffolding

The ICR requires cash-flow decomposition:
- **Intercepted flow** = rent + mortgage payments + consumer debt service (money leaving the household without building equity)
- **Compounding flow** = savings deposits + asset purchases + equity paydown (money building durable wealth)
- **ICR** = compounding_flow / max(intercepted_flow + compounding_flow, ε)

Neither the PUMS person-level nor household-level file contains these variables. Required data sources:
- **SCF (Survey of Consumer Finances)**: `X7575` (mortgage payment), `X1055` (rent), credit card balances, installment debt service.
- **Consumer Expenditure Survey (CEX)**: FMLI file contains `MRPINC` (mortgage principal), `MRTINT` (mortgage interest), `RENTX` (rent).

The function below is a structural scaffold. Call it with a DataFrame that has been enriched with `intercepted_flow` and `compounding_flow` columns (joined from SCF/CEX on race/income-decile strata).

In [7]:
# Cell 3.1 — ICR proxy scaffold

def estimate_icr_proxy(df_households: pd.DataFrame) -> pd.DataFrame:
    """ICR scaffold.

    intercepted_flow and compounding_flow are zero placeholders.
    Populate by joining SCF or CEX data before calling.

    Expected enrichment columns (before call):
        intercepted_flow : rent + mortgage_payment + debt_service  [$/year]
        compounding_flow : savings_deposits + asset_purchases       [$/year]
    """
    out = df_households[["SERIALNO", "hh_income", "hh_race_ref", "equiv_income"]].copy()

    if "intercepted_flow" not in df_households.columns:
        out["intercepted_flow"] = out["hh_income"] * 0  # placeholder
    else:
        out["intercepted_flow"] = df_households["intercepted_flow"]

    if "compounding_flow" not in df_households.columns:
        out["compounding_flow"] = out["hh_income"] * 0  # placeholder
    else:
        out["compounding_flow"] = df_households["compounding_flow"]

    total_flow = out["intercepted_flow"] + out["compounding_flow"]
    # ICR undefined when both flows are zero (placeholder state)
    out["icr"] = np.where(
        total_flow.abs() < 1.0,
        np.nan,
        out["compounding_flow"] / total_flow,
    )
    return out


icr_df = estimate_icr_proxy(hh)
print("ICR scaffold output (placeholder state \u2014 flows are zero, ICR is NaN):")
print(icr_df[["SERIALNO", "hh_income", "hh_race_ref", "intercepted_flow", "compounding_flow", "icr"]].head(8))
print()
print(f"icr NaN fraction: {icr_df['icr'].isna().mean():.1%}  (expected 100% in placeholder state)")

ICR scaffold output (placeholder state — flows are zero, ICR is NaN):
        SERIALNO  hh_income hh_race_ref  intercepted_flow  compounding_flow  \
0  2024GQ0000205     4300.0       White               0.0               0.0   
1  2024GQ0000221    10000.0       Black               0.0               0.0   
2  2024GQ0000421     5000.0       Black               0.0               0.0   
3  2024GQ0001026     6800.0       White               0.0               0.0   
4  2024GQ0001246        0.0       Black               0.0               0.0   
5  2024GQ0001674    18500.0       White               0.0               0.0   
6  2024GQ0002139      660.0       White               0.0               0.0   
7  2024GQ0003212        0.0       White               0.0               0.0   

   icr  
0  NaN  
1  NaN  
2  NaN  
3  NaN  
4  NaN  
5  NaN  
6  NaN  
7  NaN  

icr NaN fraction: 100.0%  (expected 100% in placeholder state)


---
## Section 4 — Sheaf Gluing (Overlap Consistency)

The sheaf model represents each racial subgroup as a **local section** — a vector of summary statistics over its income distribution. The **overlap** between two sections is the common support: households that belong to both the subgroup and the overall population.

The `sheaf_status_and_kappa` function (Curry 2014 cellular sheaf cocycle; axioms §4) checks whether local sections agree on their shared support within tolerance `tol`. A single overlap violation is sufficient to mark the sheaf **obstructed** — no averaging, no voting.

**Sections:** `[mean, var, skew, kurt]` of equivalised income for (Black, White, All) households above the nucleation threshold.  
**Overlaps:** `[[0, 2], [1, 2]]` — Black↔All and White↔All.  
**Normalization:** z-score across groups per statistic, so scale differences in raw income do not dominate the disagreement norm.

In [8]:
# Cell 4.1 — Build local sections [3, 4]

def income_stats(arr: np.ndarray) -> list:
    """Return [mean, var, skew, kurt] as floats."""
    return [
        float(np.mean(arr)),
        float(np.var(arr)),
        float(stats.skew(arr)),
        float(stats.kurtosis(arr)),
    ]

black_inc = hh_above[hh_above["hh_race_ref"] == "Black"]["equiv_income"].values
white_inc = hh_above[hh_above["hh_race_ref"] == "White"]["equiv_income"].values
all_inc   = hh_above["equiv_income"].values

for label, arr in [("Black", black_inc), ("White", white_inc), ("All", all_inc)]:
    if len(arr) < 4:
        raise RuntimeError(f"Subgroup '{label}' has fewer than 4 samples ({len(arr)}); cannot compute kurtosis.")

raw_sections = np.array([
    income_stats(black_inc),
    income_stats(white_inc),
    income_stats(all_inc),
])  # [3, 4]

print("Raw sections [mean, var, skew, kurt]:")
for label, row in zip(["Black", "White", "All"], raw_sections):
    print(f"  {label:<6}: {row}")
print()

# Z-score normalize per statistic column so scale differences don't dominate
col_mean = raw_sections.mean(axis=0)
col_std  = raw_sections.std(axis=0).clip(min=1e-8)
normed   = (raw_sections - col_mean) / col_std

sections = torch.tensor(normed, dtype=torch.float64)           # [3, 4]
overlaps  = torch.tensor([[0, 2], [1, 2]], dtype=torch.long)   # Black\u2194All, White\u2194All

print("Normalized sections (z-score per column):")
for label, row in zip(["Black", "White", "All"], normed):
    print(f"  {label:<6}: {row.tolist()}")

Raw sections [mean, var, skew, kurt]:
  Black : [7.89999566e+04 3.74690392e+09 4.52732573e+00 2.92256318e+01]
  White : [9.95571260e+04 6.43377687e+09 3.24562509e+00 1.50359161e+01]
  All   : [9.77634484e+04 6.28418776e+09 3.29441866e+00 1.53799360e+01]

Normalized sections (z-score per column):
  Black : [-1.4098199084005556, -1.4124774673422145, 1.4134155246804856, 1.4138942524065177]
  White : [0.8013768884735601, 0.7669062965683703, -0.7478467456060006, -0.732971773345162]
  All   : [0.608443019926997, 0.6455711707738426, -0.6655687790744858, -0.6809224790613558]


In [9]:
# Cell 4.2 — Sheaf gluing check
#
# tol=0.5 in z-score units: half a standard deviation of disagreement triggers obstruction.
# Conformance: Curry 2014 cellular sheaf cocycle equality (axioms §4).

status, kappa_residual, obstruction_at = sheaf_status_and_kappa(sections, overlaps, tol=0.5)

print(f"sheaf_status    : {status}")
print(f"kappa_residual  : {kappa_residual}")
print(f"obstruction_at  : {obstruction_at}")

sheaf_status    : obstructed
kappa_residual  : [4.125455407395643, 0.24783963201215356]
obstruction_at  : sheaf:overlap_0_2


### Sheaf Result Interpretation

| Tag | Meaning |
|---|---|
| `sheaf:overlap_0_2` | Black subgroup section disagrees with the All-population section beyond tolerance — Black households are **structurally separated** from the aggregate distribution |
| `sheaf:overlap_1_2` | White subgroup section disagrees with the All-population section — White households diverge from the aggregate (less likely if White is the dominant group) |
| `None` (status=`valid`) | All sections agree within `tol=0.5σ` — no structural impedance detectable at this statistical resolution |

**Mathematical fingerprint:** Obstruction at `overlap_0_2` is the sheaf-theoretic encoding of the Chile-Barbados structural impedance: the Black income distribution's higher-order shape (skew, kurtosis) is inconsistent with what the aggregate distribution predicts for that subgroup. This is the prerequisite condition for wealth-channel interception (ICR > 0 on the Black-household stratum).

**Next step:** Join `psam_h10.csv` (household-level PUMS) on `SERIALNO` to add `VALP`, `TEN`, `HINCP`. Populate `estimate_icr_proxy` with CEX/SCF rent and debt-service data to compute non-zero ICR values per household.